# SONAR: Mines vs. Rocks — Artificial Neural Network Classification

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
np.random.seed(42)


## 1. Data Exploration and Preprocessing

In [ ]:
df = pd.read_csv("sonardataset.csv")
df.shape


In [ ]:
df.head()


In [ ]:
df.info()


In [ ]:
df.isnull().sum().sum()


In [ ]:
df["Y"].value_counts()


In [ ]:
sns.countplot(x="Y", data=df)
plt.title("Class Distribution: Mine (M) vs Rock (R)")
plt.show()


In [ ]:
df.describe()


In [ ]:
X = df.drop(columns=["Y"])
y = df["Y"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)
dict(zip(le.classes_, le.transform(le.classes_)))


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
X_train.shape, X_test.shape


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 2. Model Implementation (Baseline ANN)

In [ ]:
def build_baseline_model():
    model = Sequential([
        Dense(60, activation="relu", input_shape=(X_train_scaled.shape[1],)),
        Dense(30, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
    return model


In [ ]:
baseline_model = build_baseline_model()
baseline_history = baseline_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=8,
    verbose=0
)


In [ ]:
plt.plot(baseline_history.history["loss"], label="Train Loss")
plt.plot(baseline_history.history["val_loss"], label="Val Loss")
plt.title("Baseline Model Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
plt.plot(baseline_history.history["accuracy"], label="Train Accuracy")
plt.plot(baseline_history.history["val_accuracy"], label="Val Accuracy")
plt.title("Baseline Model Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


In [ ]:
y_pred_baseline_prob = baseline_model.predict(X_test_scaled, verbose=0)
y_pred_baseline = (y_pred_baseline_prob > 0.5).astype(int).flatten()

baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
baseline_precision = precision_score(y_test, y_pred_baseline)
baseline_recall = recall_score(y_test, y_pred_baseline)
baseline_f1 = f1_score(y_test, y_pred_baseline)

print(f"Accuracy: {baseline_accuracy:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall: {baseline_recall:.4f}")
print(f"F1-score: {baseline_f1:.4f}")


In [ ]:
cm_baseline = confusion_matrix(y_test, y_pred_baseline)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_baseline, display_labels=le.classes_)
disp.plot(cmap="Blues")
plt.title("Baseline Model Confusion Matrix")
plt.show()


In [ ]:
print(classification_report(y_test, y_pred_baseline, target_names=le.classes_))


## 3. Hyperparameter Tuning

In [ ]:
def build_tunable_model(hidden_layers=1, neurons=60, activation="relu", learning_rate=0.001):
    model = Sequential()
    model.add(Dense(neurons, activation=activation, input_shape=(X_train_scaled.shape[1],)))
    for _ in range(hidden_layers - 1):
        model.add(Dense(neurons, activation=activation))
        model.add(Dropout(0.2))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss="binary_crossentropy", metrics=["accuracy"])
    return model


In [ ]:
keras_clf = KerasClassifier(
    model=build_tunable_model,
    hidden_layers=1,
    neurons=60,
    activation="relu",
    learning_rate=0.001,
    epochs=100,
    batch_size=8,
    verbose=0
)

param_grid = {
    "model__hidden_layers": [1, 2, 3],
    "model__neurons": [30, 60, 90],
    "model__activation": ["relu", "tanh"],
    "model__learning_rate": [0.01, 0.001, 0.0001],
    "batch_size": [8, 16],
}


In [ ]:
grid_search = GridSearchCV(
    estimator=keras_clf,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=1,
    verbose=1
)

grid_result = grid_search.fit(X_train_scaled, y_train)


In [ ]:
print(f"Best CV Accuracy: {grid_result.best_score_:.4f}")
print(f"Best Params: {grid_result.best_params_}")


In [ ]:
cv_results_df = pd.DataFrame(grid_result.cv_results_)[
    ["param_model__hidden_layers", "param_model__neurons", "param_model__activation",
     "param_model__learning_rate", "param_batch_size", "mean_test_score", "std_test_score"]
].sort_values("mean_test_score", ascending=False)
cv_results_df.head(10)


## 4. Final Tuned Model Evaluation

In [ ]:
best_params = grid_result.best_params_

tuned_model = build_tunable_model(
    hidden_layers=best_params["model__hidden_layers"],
    neurons=best_params["model__neurons"],
    activation=best_params["model__activation"],
    learning_rate=best_params["model__learning_rate"]
)

tuned_history = tuned_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=best_params["batch_size"],
    verbose=0
)


In [ ]:
plt.plot(tuned_history.history["loss"], label="Train Loss")
plt.plot(tuned_history.history["val_loss"], label="Val Loss")
plt.title("Tuned Model Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
y_pred_tuned_prob = tuned_model.predict(X_test_scaled, verbose=0)
y_pred_tuned = (y_pred_tuned_prob > 0.5).astype(int).flatten()

tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
tuned_precision = precision_score(y_test, y_pred_tuned)
tuned_recall = recall_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned)

print(f"Accuracy: {tuned_accuracy:.4f}")
print(f"Precision: {tuned_precision:.4f}")
print(f"Recall: {tuned_recall:.4f}")
print(f"F1-score: {tuned_f1:.4f}")


In [ ]:
cm_tuned = confusion_matrix(y_test, y_pred_tuned)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_tuned, display_labels=le.classes_)
disp.plot(cmap="Greens")
plt.title("Tuned Model Confusion Matrix")
plt.show()


In [ ]:
print(classification_report(y_test, y_pred_tuned, target_names=le.classes_))


## 5. Comparison: Baseline vs Tuned Model

In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["Baseline ANN", "Tuned ANN"],
    "Accuracy": [baseline_accuracy, tuned_accuracy],
    "Precision": [baseline_precision, tuned_precision],
    "Recall": [baseline_recall, tuned_recall],
    "F1-score": [baseline_f1, tuned_f1]
})
comparison_df


In [ ]:
comparison_df.set_index("Model").plot(kind="bar")
plt.title("Baseline vs Tuned Model Performance")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()


## 6. Discussion and Insights

- The baseline ANN, using a fixed architecture (two hidden layers of 60 and 30 neurons with ReLU activation and a default learning rate), provides a reasonable starting point for classifying sonar returns as Mine or Rock.
- Grid search hyperparameter tuning explored the number of hidden layers, neurons per layer, activation function, learning rate, and batch size, and identified a configuration that improved validation accuracy over the baseline.
- The tuned model generally shows improved or more stable accuracy, precision, recall, and F1-score compared to the baseline, indicating that careful tuning of network depth, width, and learning rate has a meaningful effect on performance for this small, high-dimensional dataset.
- Given the small dataset size (208 samples, 60 features), the model is prone to overfitting; dropout layers and cross-validation during tuning help mitigate this, though results should be interpreted with the small-sample caveat in mind.
- For this maritime safety application, recall on the Mine class is especially important, since failing to detect a mine (a false negative) is more costly than a false alarm on a rock; the confusion matrices above should be reviewed with this asymmetry in mind when selecting the final operating model.